# 📧 Email Producer for Phishing Detection Pipeline

This notebook reads emails from a dataset and streams them to Kafka for real-time processing.

**Part of**: Phishing Email Detection Pipeline Project

**MITRE ATT&CK Coverage**: T1566 (Phishing), T1598 (Phishing for Information)

In [1]:
!pip install kafka-python jaeger-client pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.4/87.4 kB 917.4 kB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 326.3/326.3 kB 3.0 MB/s eta 0:00:00a 0:00:01
  Created wheel for jaeger-client: filename=jaeger_client-4.8.0-py3-none-any.whl size=66029 sha256=2c36bd8f4c7077729c50d9fa488c74d8e0ed2b54f174584a704e13b725e241a9
  Stored in directory: /home/jovyan/.cache/pip/wheels/44/c6/51/bd6a454e0a5f8ce568c515446aa30f5a4e6fbb575e34a1ca76
  Created wheel for opentracing: filename=opentracing-2.4.0-py3-none-any.whl size=51404 sha256=d52e21cf6861ccce480ca3869f7bc741b6cf411a3146c9c592988d0fe839a1a5
  Stored in directory: /home/jovyan/.cache/pip/wheels/d5/9a

In [2]:
import json
import time
import pandas as pd
from datetime import datetime
from kafka import KafkaProducer
from jaeger_client import Config
import uuid

In [3]:
# Initialize Jaeger tracer for distributed tracing
config = Config(
    config={
        'sampler': {'type': 'const', 'param': 1},
        'logging': True,
        'local_agent': {
            'reporting_host': 'jaeger',
            'reporting_port': 6831,
        },
    },
    service_name='email-producer',
    validate=True,
)
tracer = config.initialize_tracer()
print("✅ Jaeger tracer initialized!")

✅ Jaeger tracer initialized!


In [4]:
# Initialize Kafka producer
producer = KafkaProducer(
    bootstrap_servers=['kafka:9092'],
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)
print("✅ Kafka producer connected!")

✅ Kafka producer connected!


In [5]:
# Load email dataset
df = pd.read_csv('/home/jovyan/data/emails.csv')
print(f"📊 Loaded {len(df)} emails from dataset")
print(f"\nLabel distribution:")
print(df['label'].value_counts())
print(f"\n📧 Sample email:")
print(df.iloc[0])

📊 Loaded 100 emails from dataset

Label distribution:
label
legitimate    63
phishing      37
Name: count, dtype: int64

📧 Sample email:
email_id                                                    1
subject             Urgent: Your account has been compromised
body        Dear Customer, We detected suspicious activity...
sender                       security@bankofamerica-secure.tk
label                                                phishing
Name: 0, dtype: object


In [6]:
def produce_emails(dataframe, delay=0.1):
    """
    Stream emails to Kafka topic 'emails.raw'
    
    Args:
        dataframe: DataFrame containing emails
        delay: Delay between messages (simulates real-time stream)
    """
    print(f"📤 Streaming {len(dataframe)} emails to Kafka...\n")
    
    for idx, row in dataframe.iterrows():
        with tracer.start_span('produce_email') as span:
            # Create email event
            email_event = {
                'event_id': str(uuid.uuid4()),
                'timestamp': datetime.utcnow().isoformat(),
                'email_id': int(row['email_id']),
                'subject': row['subject'],
                'body': row['body'],
                'sender': row['sender'],
                'original_label': row['label']  # We keep this for evaluation
            }
            
            span.set_tag('event_id', email_event['event_id'])
            span.set_tag('email_id', email_event['email_id'])
            
            # Send to Kafka
            with tracer.start_span('kafka_send', child_of=span) as send_span:
                producer.send('emails.raw', value=email_event)
                send_span.set_tag('topic', 'emails.raw')
            
            if (idx + 1) % 20 == 0:
                print(f"📧 Sent {idx + 1} emails...")
            
            time.sleep(delay)
    
    # Flush to ensure all messages are sent
    producer.flush()
    
    print(f"\n✅ Successfully sent {len(dataframe)} emails to Kafka!")
    print(f"🔗 View messages at: http://localhost:8080")
    print(f"🔍 View traces at: http://localhost:16686")

In [7]:
# Run the producer
produce_emails(df, delay=0.05)

📤 Streaming 100 emails to Kafka...

📧 Sent 20 emails...
📧 Sent 40 emails...
📧 Sent 60 emails...
📧 Sent 80 emails...
📧 Sent 100 emails...

✅ Successfully sent 100 emails to Kafka!
🔗 View messages at: http://localhost:8080
🔍 View traces at: http://localhost:16686


In [8]:
# Summary statistics
print("\n" + "="*50)
print("📊 EMAIL PRODUCTION SUMMARY")
print("="*50)
print(f"Total emails sent: {len(df)}")
print(f"Phishing emails: {len(df[df['label']=='phishing'])}")
print(f"Legitimate emails: {len(df[df['label']=='legitimate'])}")
print(f"\nKafka Topic: emails.raw")
print(f"Next step: Run 2_ML_Classifier.ipynb to process emails")


📊 EMAIL PRODUCTION SUMMARY
Total emails sent: 100
Phishing emails: 37
Legitimate emails: 63

Kafka Topic: emails.raw
Next step: Run 2_ML_Classifier.ipynb to process emails
